# BCI Calibration Study (builds on BCI_Tier1_Pipeline)
**Question:** How many labeled trials does a *new* user need before an EEGNet motor-imagery decoder works,
and does Euclidean Alignment (EA) reduce that number? How does transfer learning compare with
simply calibrating a classic CSP+LDA decoder on the same few trials?

**Protocol (fixed before running):**
- Data: BCI Competition IV-2a (BNCI2014-001), 9 subjects, 8 motor channels, 3 classes (left / right / feet).
- Leave-one-subject-out: pretrain on the other 8 subjects, adapt to the target subject.
- **Session split (new, stricter than Tier 1):** calibration trials come ONLY from the target's session 1.
  All test trials come from the target's session 2 (recorded on a different day). Test data is never used for
  fitting, normalisation or alignment.
- Calibration size k = 0, 5, 10, 20, 40 labeled trials per class. Nested random draws, repeated.
- Alignment uses 48 UNLABELED trials from the start of session 1 (a short setup recording). This is stated openly.
- Methods: EEGNet without alignment, EEGNet + EA, CSP+LDA trained only on the k calibration trials.

**Hypotheses (written before the run):**
- H1: EA helps most when k is small.
- H2: Subjects that are weak at k=0 stay weak at k=40.
- Exploratory only: how many of the 9 users are above the statistical chance level at each k.

Run with `QUICK = True` first (smoke test, few minutes). Then set `QUICK = False` for the real study.

In [ ]:
!pip install -q moabb mne torch scikit-learn scipy pandas matplotlib

In [ ]:
import os, copy, time, math, json
import numpy as np, pandas as pd
import torch, torch.nn as nn
from scipy.signal import butter, filtfilt
from scipy.stats import wilcoxon, spearmanr, binom

# ---------------- CONFIG ----------------
QUICK = True            # True: 2 target subjects, 1 seed, 5 pretrain epochs (smoke test). False: full study.
DATA = "real"           # "real" = BNCI2014-001 via MOABB, "synthetic" = fake data to test the code
CLASSES = ("left_hand", "right_hand", "feet")
CHANNELS = ("FC3", "FC1", "FCz", "FC2", "FC4", "C3", "Cz", "C4")
ALL_SUBJECTS = list(range(1, 10))

if QUICK:
    TARGETS, SEEDS, PRETRAIN_EPOCHS, DRAWS = [1, 2], [0], 5, 2
else:
    TARGETS, SEEDS, PRETRAIN_EPOCHS, DRAWS = ALL_SUBJECTS, [0, 1, 2], 60, 5

KS = [0, 5, 10, 20, 40]     # labeled trials per class
PRETRAIN_LR, FINETUNE_LR = 1e-3, 1e-4
BATCH, FINETUNE_STEPS = 16, 100
N_REF = 48                  # unlabeled trials used for alignment / normalisation of the target
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "calibstudy_out"
os.makedirs(OUT_DIR, exist_ok=True)
TAG = "quick" if QUICK else "full"
RESULTS_CSV = f"{OUT_DIR}/results_{TAG}.csv"
print("device:", DEVICE, "| quick:", QUICK, "| targets:", TARGETS, "| seeds:", SEEDS)

In [ ]:
# ---------------- EEGNet (same architecture as your Tier 1 notebook) ----------------
class EEGNet(nn.Module):
    def __init__(self, n_classes=3, n_channels=8, n_samples=1000,
                 F1=8, D=2, F2=16, kernel_length=125, dropout_rate=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout_rate))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout_rate))
        with torch.no_grad():
            flat = self.block2(self.block1(torch.zeros(1, 1, n_channels, n_samples))).numel()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(flat, n_classes))

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

In [ ]:
# ---------------- Data ----------------
def load_real(subjects):
    import moabb.datasets as mdatasets
    from moabb.paradigms import MotorImagery
    ds = mdatasets.BNCI2014_001() if hasattr(mdatasets, "BNCI2014_001") else mdatasets.BNCI2014001()
    paradigm = MotorImagery(channels=list(CHANNELS), events=list(CLASSES), n_classes=len(CLASSES))
    label_map = {c: i for i, c in enumerate(CLASSES)}
    data = {}
    for sid in subjects:
        X, labels, meta = paradigm.get_data(dataset=ds, subjects=[sid])
        y = np.array([label_map[l] for l in labels], dtype=np.int64)
        names = list(dict.fromkeys(meta["session"].tolist()))      # order of appearance
        assert len(names) == 2, f"expected 2 sessions, got {names}"
        sess = np.array([names.index(s) for s in meta["session"].tolist()])
        data[sid] = dict(X=X.astype(np.float32), y=y, sess=sess)
        print(f"subject {sid}: X={X.shape}, sessions {names} -> 0/1, trials per session {np.bincount(sess)}")
    return data


def make_synthetic(subjects, per_class_per_session=50, n_ch=8, n_samples=1001, fs=250, seed=0):
    rng = np.random.RandomState(seed)
    t = np.arange(n_samples) / fs
    class_channel = {0: 5, 1: 7, 2: 6}          # C3, C4, Cz carry the class signal
    data = {}
    for sid in subjects:
        Xs, ys, ss = [], [], []
        strength = 0.8 + 1.2 * rng.rand()        # some subjects are weaker than others
        for s in (0, 1):
            for c in range(3):
                for _ in range(per_class_per_session):
                    x = rng.randn(n_ch, n_samples).astype(np.float32)
                    x[class_channel[c]] += strength * np.sin(2 * np.pi * 10 * t + rng.rand() * 6.28)
                    Xs.append(x); ys.append(c); ss.append(s)
        Xs, ys, ss = np.stack(Xs), np.array(ys), np.array(ss)
        order = np.concatenate([rng.permutation(np.where(ss == s)[0]) for s in (0, 1)])
        data[sid] = dict(X=Xs[order], y=ys[order], sess=ss[order])
    return data


RAW = load_real(ALL_SUBJECTS) if DATA == "real" else make_synthetic(ALL_SUBJECTS)

In [ ]:
# ---------------- Preprocessing: bandpass, normalisation, Euclidean Alignment ----------------
def bandpass(X, low=8.0, high=30.0, fs=250.0, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, X, axis=-1).astype(np.float32)


def zscore_from_ref(X, Xref):
    m = Xref.mean(axis=(0, 2), keepdims=True)
    s = Xref.std(axis=(0, 2), keepdims=True) + 1e-8
    return ((X - m) / s).astype(np.float32)


def ea_matrix(Xref):
    R = np.mean([x @ x.T / x.shape[1] for x in Xref], axis=0)
    w, V = np.linalg.eigh(R)
    return (V @ np.diag(np.clip(w, 1e-10, None) ** -0.5) @ V.T).astype(np.float32)


def align(X, Xref, mode):
    if mode == "ea":
        return np.einsum("ij,njt->nit", ea_matrix(Xref), X).astype(np.float32)
    return zscore_from_ref(X, Xref)


FILT = {sid: dict(X=bandpass(d["X"]), y=d["y"], sess=d["sess"]) for sid, d in RAW.items()}
MODES = ["none", "ea"]
SRC, TGT = {m: {} for m in MODES}, {m: {} for m in MODES}
for sid, d in FILT.items():
    s0 = np.where(d["sess"] == 0)[0]
    ref = d["X"][s0[:min(N_REF, len(s0))]]          # first unlabeled trials of session 1
    for m in MODES:
        SRC[m][sid] = align(d["X"], d["X"], m)       # as a source subject: all its data
        TGT[m][sid] = align(d["X"], ref, m)          # as the target: only the unlabeled reference
print("preprocessing done")

In [ ]:
# ---------------- Training utilities ----------------
def pretrain(Xs, ys, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = EEGNet(len(CLASSES), Xs.shape[1], Xs.shape[2]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=PRETRAIN_LR)
    crit = nn.CrossEntropyLoss()
    Xt = torch.from_numpy(Xs).unsqueeze(1).to(DEVICE)
    yt = torch.from_numpy(ys).to(DEVICE)
    n = len(yt)
    for _ in range(PRETRAIN_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            crit(model(Xt[idx]), yt[idx]).backward()
            opt.step()
    return model


@torch.no_grad()
def accuracy(model, X, y, chunk=128):
    model.eval()
    correct = 0
    for i in range(0, len(y), chunk):
        xb = torch.from_numpy(X[i:i + chunk]).unsqueeze(1).to(DEVICE)
        correct += int((model(xb).argmax(1).cpu().numpy() == y[i:i + chunk]).sum())
    return correct / len(y)


def finetune(base, Xc, yc, seed):
    """Fixed budget of optimizer steps, so different k are not confounded by training length."""
    torch.manual_seed(seed)
    m = copy.deepcopy(base)
    opt = torch.optim.Adam(m.parameters(), lr=FINETUNE_LR)
    crit = nn.CrossEntropyLoss()
    Xt = torch.from_numpy(Xc).unsqueeze(1).to(DEVICE)
    yt = torch.from_numpy(yc).to(DEVICE)
    n, bs = len(yt), min(BATCH, len(yt))
    m.train()
    for _ in range(FINETUNE_STEPS):
        idx = torch.randint(0, n, (bs,), device=DEVICE)
        opt.zero_grad()
        crit(m(Xt[idx]), yt[idx]).backward()
        opt.step()
    return m


def calib_indices(y_s0, k, perms):
    """Nested draws: the k=5 set is inside the k=10 set, and so on."""
    return np.concatenate([perms[c][:k] for c in range(len(CLASSES))])


def make_perms(y_s0, sid, draw):
    rng = np.random.RandomState(1000 * sid + draw)
    return {c: rng.permutation(np.where(y_s0 == c)[0]) for c in range(len(CLASSES))}

In [ ]:
# ---------------- CSP + LDA baseline (within-subject, only the k calibration trials) ----------------
import mne
mne.set_log_level('ERROR')
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import make_pipeline


def csp_lda_acc(Xc, yc, Xt, yt):
    try:
        clf = make_pipeline(CSP(n_components=6, reg="oas", log=True), LDA())
        clf.fit(Xc.astype(np.float64), yc)
        return float((clf.predict(Xt.astype(np.float64)) == yt).mean())
    except Exception as e:
        print("CSP+LDA failed:", repr(e)[:120])
        return float("nan")

In [ ]:
# ---------------- Main loop (resumable: re-run the cell after a disconnect) ----------------
done = set()
rows_all = []
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    rows_all = prev.to_dict("records")
    done = set(zip(prev.method, prev.subject, prev.seed))


def save():
    pd.DataFrame(rows_all).to_csv(RESULTS_CSV, index=False)


t0 = time.time()
for sid in TARGETS:
    d = FILT[sid]
    s0 = np.where(d["sess"] == 0)[0]
    s1 = np.where(d["sess"] == 1)[0]
    y_s0, y_test = d["y"][s0], d["y"][s1]
    per_class_avail = min(np.bincount(y_s0, minlength=len(CLASSES)))
    ks = [k for k in KS if k <= per_class_avail]

    # --- CSP+LDA baseline
    if ("CSP+LDA", sid, -1) not in done:
        for draw in range(DRAWS):
            perms = make_perms(y_s0, sid, draw)
            for k in ks:
                if k == 0:
                    continue
                ci = calib_indices(y_s0, k, perms)
                acc = csp_lda_acc(d["X"][s0][ci], y_s0[ci], d["X"][s1], y_test)
                rows_all.append(dict(method="CSP+LDA", subject=sid, seed=-1, draw=draw, k=k, acc=acc))
        save()

    # --- EEGNet, with and without alignment
    for mode in MODES:
        for seed in SEEDS:
            name = f"EEGNet-{mode}"
            if (name, sid, seed) in done:
                continue
            src_ids = [s for s in ALL_SUBJECTS if s != sid]
            Xs = np.concatenate([SRC[mode][s] for s in src_ids])
            ys = np.concatenate([FILT[s]["y"] for s in src_ids])
            base = pretrain(Xs, ys, seed)
            Xtgt = TGT[mode][sid]
            Xc_pool, Xtest = Xtgt[s0], Xtgt[s1]
            rows_all.append(dict(method=name, subject=sid, seed=seed, draw=0, k=0,
                                 acc=accuracy(base, Xtest, y_test)))
            for draw in range(DRAWS):
                perms = make_perms(y_s0, sid, draw)
                for k in ks:
                    if k == 0:
                        continue
                    ci = calib_indices(y_s0, k, perms)
                    m = finetune(base, Xc_pool[ci], y_s0[ci], seed=seed * 100 + draw)
                    rows_all.append(dict(method=name, subject=sid, seed=seed, draw=draw, k=k,
                                         acc=accuracy(m, Xtest, y_test)))
            save()
            last = [r for r in rows_all if r["method"] == name and r["subject"] == sid and r["k"] == 0][-1]
            print(f"subject {sid} {name} seed {seed}: zero-shot {last['acc']*100:.1f}%  "
                  f"({(time.time()-t0)/60:.1f} min elapsed)")
save()
print("finished ->", RESULTS_CSV)

In [ ]:
# ---------------- Analysis ----------------
import matplotlib
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)
n_test = int((FILT[TARGETS[0]]["sess"] == 1).sum())
chance = 1 / len(CLASSES)
sig_thr = binom.ppf(0.95, n_test, chance) / n_test      # accuracy needed to beat chance at p<0.05
print(f"test trials per subject: {n_test} | chance {chance*100:.1f}% | significance threshold {sig_thr*100:.1f}%")

per_subj = df.groupby(["method", "subject", "k"]).acc.mean().reset_index()
rng = np.random.RandomState(0)


def boot_ci(v, n=10000):
    v = np.asarray(v)
    m = [rng.choice(v, len(v)).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])


rows = []
for (method, k), g in per_subj.groupby(["method", "k"]):
    lo, hi = boot_ci(g.acc.values)
    rows.append(dict(method=method, k=k, mean=g.acc.mean() * 100, ci_low=lo * 100, ci_high=hi * 100,
                     n_subjects=len(g), users_above_chance=int((g.acc > sig_thr).sum())))
summary = pd.DataFrame(rows).sort_values(["method", "k"])
print(summary.round(1).to_string(index=False))
summary.to_csv(f"{OUT_DIR}/summary_{TAG}.csv", index=False)

# H1: EA vs no alignment, paired across subjects
print("\nH1: EEGNet-EA minus EEGNet-none (paired by subject)")
for k in sorted(per_subj.k.unique()):
    a = per_subj[(per_subj.method == "EEGNet-ea") & (per_subj.k == k)].set_index("subject").acc
    b = per_subj[(per_subj.method == "EEGNet-none") & (per_subj.k == k)].set_index("subject").acc
    common = a.index.intersection(b.index)
    if len(common) < 6:
        print(f"  k={k}: only {len(common)} subjects, test skipped"); continue
    diff = (a[common] - b[common]) * 100
    p = wilcoxon(diff).pvalue if (diff != 0).any() else 1.0
    print(f"  k={k:>2}: mean diff {diff.mean():+.2f} pp, Wilcoxon p={p:.3f}, n={len(common)}")

# H2: do weak users stay weak?
best = "EEGNet-ea"
lo_k, hi_k = 0, max(KS)
a = per_subj[(per_subj.method == best) & (per_subj.k == lo_k)].set_index("subject").acc
b = per_subj[(per_subj.method == best) & (per_subj.k == hi_k)].set_index("subject").acc
common = a.index.intersection(b.index)
if len(common) >= 5:
    r, p = spearmanr(a[common], b[common])
    print(f"\nH2 ({best}): Spearman rho between k=0 and k={hi_k} accuracy = {r:.2f} (p={p:.3f}, n={len(common)})")
    print("  (with 9 subjects this is exploratory, not proof)")

# Figure 1: calibration curve
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = {"EEGNet-none": "#1f77b4", "EEGNet-ea": "#d62728", "CSP+LDA": "#2ca02c"}
for method, g in summary.groupby("method"):
    ax.errorbar(g.k, g["mean"], yerr=[g["mean"] - g.ci_low, g.ci_high - g["mean"]],
                marker="o", capsize=3, label=method, color=colors.get(method))
ax.axhline(chance * 100, ls=":", color="gray", label="chance")
ax.axhline(sig_thr * 100, ls="--", color="gray", label="p<0.05 above chance")
ax.set_xlabel("Labeled calibration trials per class (new user)")
ax.set_ylabel("Accuracy on later session (%)")
ax.set_title("How much calibration does a new user need?")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig1_calibration_curve_{TAG}.png", dpi=200)

# Figure 2: every subject
fig, ax = plt.subplots(figsize=(7, 4.5))
for sid, g in per_subj[per_subj.method == best].groupby("subject"):
    ax.plot(g.k, g.acc * 100, marker=".", alpha=0.7, label=f"S{sid}")
ax.axhline(sig_thr * 100, ls="--", color="gray")
ax.set_xlabel("Labeled calibration trials per class"); ax.set_ylabel("Accuracy (%)")
ax.set_title(f"Every subject, {best}"); ax.legend(ncol=3, fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/fig2_per_subject_{TAG}.png", dpi=200)
plt.show()

In [ ]:
# ---------------- Save to Google Drive (optional, recommended) ----------------
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r calibstudy_out /content/drive/MyDrive/calibstudy_out